# **MeetingMind AI Agent: A Retrieval-Augmented Meeting Assistant**

**An Intelligent Meeting Assistant Powered by Generative AI and Retrieval-Augmented Generation (RAG)**

**Capstone Project**


Project Type: Single-Agent AI Application

**Project Overview**

MeetingMind AI Agent is an intelligent meeting assistant that helps users analyze meeting transcripts and notes using a Large Language Model (LLM). The application automatically generates structured outputs such as executive summaries, key decisions, action items, follow-up emails, and suggested agendas for future meetings.

The system also allows users to ask questions about uploaded meeting documents through Retrieval-Augmented Generation (RAG), ensuring responses are based on the meeting content rather than general knowledge.

**Problem Statement**

Meetings often generate lengthy notes and transcripts that take time to review. Identifying important decisions, assigned responsibilities, deadlines, and follow-up actions can be tedious and may result in important information being overlooked.

**Proposed Solution**

MeetingMind AI Agent simplifies meeting management by using Artificial Intelligence to analyze meeting transcripts and generate accurate, structured, and actionable outputs. The application combines a Large Language Model with Retrieval-Augmented Generation (RAG) to answer questions based on uploaded meeting documents.

**Project Objectives**

Build a single-agent AI application.
Integrate a Large Language Model (LLM).
Allow users to upload meeting transcripts.
Implement Retrieval-Augmented Generation (RAG).
Generate meeting summaries and action items.
Produce follow up emails and suggested next meeting agendas.
Deploy the application using Streamlit.

**Technologies Used**

Python

Google Colab

LangChain

Chroma Vector Database (ChromaDB)

Sentence Transformers

Streamlit

GitHub

### **Install Required Libraries**

In [ ]:
# Install required libraries

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q pypdf
!pip install -q langchain-text-splitters

print("All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY") # Ensure API key is loaded

client = genai.Client(api_key=GEMINI_API_KEY)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

**Step 2  Importing Libraries and Initializing the Gemini Model**

After installing the required packages in the previous step, the next task is to import the libraries that will be used throughout the project. Each library has a specific responsibility in the Retrieval-Augmented Generation (RAG) pipeline.

The project begins by importing Python's built-in os module, which provides access to operating system functions when needed. The userdata and files modules from Google Colab are also imported. The userdata module is used to securely retrieve the Gemini API key without exposing it in the notebook, while the files module allows users to upload meeting transcript PDFs directly into the Colab environment.

Several components from the LangChain framework are then imported to simplify document processing. PyPDFLoader is responsible for reading the uploaded PDF document page by page, making it possible to extract the meeting transcript into text format. Although TextLoader is also imported to support plain text files, this project focuses primarily on PDF meeting transcripts.

In [30]:
import os
from google.colab import userdata, files

# LangChain components
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Load Gemini API key
GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY") # Corrected to use the secret name

# Create Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # Updated to a newer, available model
    google_api_key=GEMINI_API_KEY,
    temperature=0
)

print("Gemini API loaded successfully!")

Gemini API loaded successfully!


 **Step 3  Uploading the Meeting Transcript**

Upload a meeting transcript in PDF format.

Recommended documents:

- Meeting minutes
- Project meeting transcript
- Team discussion notes
- Business meeting report

In [31]:
# Upload your meeting transcript (PDF)

print("Upload a meeting transcript (PDF)...")

uploaded = files.upload()

if uploaded:
    doc_filename = list(uploaded.keys())[0]
    print(f"\nUploaded: {doc_filename}")
else:
    print("No file uploaded.")

Upload a meeting transcript (PDF)...


Saving ABC Technologies Weekly Project Meeting Transcript.pdf to ABC Technologies Weekly Project Meeting Transcript.pdf

Uploaded: ABC Technologies Weekly Project Meeting Transcript.pdf


**Step 4  Loading the Uploaded PDF Document**

After uploading the meeting transcript, the next step is to read its contents. The PyPDFLoader is used to extract the text from each page of the PDF and store it as a document object. The program also displays the total number of pages loaded and shows a preview of the first page to confirm that the document has been read successfully.

In [32]:
# Load the uploaded meeting transcript

loader = PyPDFLoader(doc_filename)

documents = loader.load()

print(f"Number of pages loaded: {len(documents)}")

# Display the first page
print("\nFirst Page Preview:\n")
print(documents[0].page_content[:1000])




Number of pages loaded: 4

First Page Preview:

ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Location: Conference Room A 
Meeting Type: Weekly Project Progress Meeting 
 
Participants 
 John Williams – Project Manager  
 Sarah Johnson – UI/UX Designer  
 David Brown – Software Developer  
 Mary Adams – Marketing Lead  
 Michael Green – Quality Assurance Engineer  
 Linda Thomas – Customer Support Manager  
 
Meeting Transcript 
 
John: Good morning, everyone. Thank you for joining today's project meeting. Our primary 
objective is to review the progress of the mobile application development, identify any 
challenges, and finalize our preparation for the product launch. 
 
Sarah: The user interface design is progressing well. I have completed approximately 80% of 
the homepage and dashboard screens. I expect to complete the remaining design work by Friday. 
After that, I will focus on improving the application's responsivene

**Step 5 Splitting the Document into Text Chunks**

Large documents are difficult for language models to process at once, so the meeting transcript is divided into smaller text chunks. A chunk size of 800 characters with an overlap of 100 characters is used to preserve context between consecutive chunks. This improves the accuracy of information retrieval during the question-answering process. The first chunk is displayed as a preview to verify that the splitting was successful.

In [33]:
# Split the meeting transcript into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

# Preview the first chunk
print("\nFirst Chunk Preview:\n")
print(chunks[0].page_content)

Total chunks created: 10

First Chunk Preview:

ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Location: Conference Room A 
Meeting Type: Weekly Project Progress Meeting 
 
Participants 
 John Williams – Project Manager  
 Sarah Johnson – UI/UX Designer  
 David Brown – Software Developer  
 Mary Adams – Marketing Lead  
 Michael Green – Quality Assurance Engineer  
 Linda Thomas – Customer Support Manager  
 
Meeting Transcript 
 
John: Good morning, everyone. Thank you for joining today's project meeting. Our primary 
objective is to review the progress of the mobile application development, identify any 
challenges, and finalize our preparation for the product launch. 
 
Sarah: The user interface design is progressing well. I have completed approximately 80% of


In [34]:
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Import Successful!")

Import Successful!


**Step 6 Creating Embeddings and Building the Vector Database**

 In this step, each text chunk is converted into a numerical representation called an embedding using the all-MiniLM-L6-v2 model. These embeddings capture the meaning of the text, making it possible to search based on context rather than exact words. The embeddings are then stored in ChromaDB, which acts as the project's vector database. This allows the system to quickly retrieve the most relevant chunks when the user asks a question.

In [35]:
print("Loading free local embedding model (all-MiniLM-L6-v2)...")
print("This downloads ~80MB the first time — takes about 30 seconds.")
print()

# Free local embedding model — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
    # Fast, accurate, widely used for RAG demos
)

print("Embedding model loaded. Now embedding chunks and storing in ChromaDB...")

# Create ChromaDB from our chunks
# This embeds EACH chunk and stores the vector + original text
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_document"
)

print()
print(f"Vector store ready!")
print(f"  {len(chunks)} chunks embedded and stored")
print(f"  Each chunk is now a vector of numbers")
print(f"  ChromaDB can find similar chunks in milliseconds")

Loading free local embedding model (all-MiniLM-L6-v2)...
This downloads ~80MB the first time — takes about 30 seconds.



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded. Now embedding chunks and storing in ChromaDB...

Vector store ready!
  10 chunks embedded and stored
  Each chunk is now a vector of numbers
  ChromaDB can find similar chunks in milliseconds


In [36]:
print(vectorstore)

In [37]:
print(llm)

metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.7'}} output_version=None profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') location=None model='gemini-3.5-flash' temperature=0.0 client=<google.genai.client.Client object at 0x7ee43c39d370> default_metadata=() model_kwargs={}


**Step 7  Test Document Retrieval**

In this step, I tested the Retrieval part of my RAG system before connecting it to Gemini. Instead of sending the entire meeting transcript to the AI model, ChromaDB searches the vector database and retrieves only the most relevant chunks related to my question. This helps confirm that the retrieval process is working correctly and that Gemini will receive only the most useful information when generating an answer.

In [38]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "What decisions were made during the meeting?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What decisions were made during the meeting?
Retrieved 3 relevant chunks:

CHUNK 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: Thank you, everyone, for your dedication and hard work. We've made excellent progress 
this week. Please complete your assigned tasks before our next meeting on Monday, July 14, 
2026, where we'll review the final launch readiness. The meeting is now adjourned.

-------------------------------------------------------
CHUNK 2:
ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Locat

In [ ]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "When is the application launch date?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: When is the application launch date?
Retrieved 3 relevant chunks:

CHUNK 1:
John: Please prepare a detailed marketing budget proposal for management before the end of 
this week. 
 
Linda: Customer support representatives will begin training next Monday. The training will 
cover common customer questions, troubleshooting procedures, and escalation processes. 
 
John: Very good. I appreciate everyone's progress. 
 
Decisions Made 
 The official application launch date remains August 1, 2026.  
 A live product demonstration will be held on July 25, 2026.  
 Final system testing will take place next Friday.  
 Customer support training begins next Monday.  
 Marketing campaign launches next Tuesday.  
 
Action Items 
 Sarah: Complete UI design by Friday and improve accessibility.  
 David: Finish payment integration, fix bugs, complete API security review.

-------------------------------------------------------
CHUNK 2:
Mary: From the marketing perspective, preparations 

In [ ]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "What action items were assigned??"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What action items were assigned??
Retrieved 3 relevant chunks:

CHUNK 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: Thank you, everyone, for your dedication and hard work. We've made excellent progress 
this week. Please complete your assigned tasks before our next meeting on Monday, July 14, 
2026, where we'll review the final launch readiness. The meeting is now adjourned.

-------------------------------------------------------
CHUNK 2:
ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Location: Confer

**Changing it to something more specific**

In [ ]:
# Change this to something specific in YOUR meeting transcript
test_question = "What decisions were made during the meeting?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Return top 3 chunks
retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What decisions were made during the meeting?
Retrieved 3 relevant chunks:

CHUNK 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: Thank you, everyone, for your dedication and hard work. We've made excellent progress 
this week. Please complete your assigned tasks before our next meeting on Monday, July 14, 
2026, where we'll review the final launch readiness. The meeting is now adjourned.

-------------------------------------------------------
CHUNK 2:
ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Locat

**Step 8  Build the RAG Question Answering Function**

In this step, I connected the retrieval system to the Gemini Large Language Model. The function first searches ChromaDB for the three most relevant chunks from the meeting transcript, combines them into a single context, and sends that context with the user's question to Gemini. This ensures that the AI answers only from the uploaded document instead of using outside knowledge, making the responses more accurate and reliable.

In [39]:
def ask_document(question, show_chunks=True):
    """Ask a question about the document using RAG + Gemini."""

    # Step 1: Retrieve relevant chunks
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5}) # Updated k to 5
    retrieved_chunks = retriever.invoke(question)

    # Step 2: Build the context string from retrieved chunks
    # Deduplicate chunks to avoid sending redundant information
    unique_chunks_content = set()
    deduplicated_chunks = []
    for chunk in retrieved_chunks:
        if chunk.page_content not in unique_chunks_content:
            unique_chunks_content.add(chunk.page_content)
            deduplicated_chunks.append(chunk)

    context = "\n\n---\n\n".join([chunk.page_content for chunk in deduplicated_chunks])

    # Step 3: Build the prompt
    user_prompt = f"""You are an AI assistant designed to extract and summarize information from meeting transcripts.
Based on the following CONTEXT from the meeting, please answer the QUESTION.
Your answer should be as comprehensive and informative as possible, drawing all relevant details and making logical inferences *only* from the provided CONTEXT.
If the CONTEXT does not contain enough information to provide a direct answer, or if the information is entirely absent, explain what information is missing or state that the answer cannot be found within the document.
Do not use any external knowledge.

CONTEXT:
{context}

QUESTION:
{question}
"""

    # Step 4: Ask Gemini
    response = llm.invoke(user_prompt)

    # Get Gemini's answer
    answer = response.content[0]["text"]

    # Print results
    print("=" * 65)
    print(f"QUESTION: {question}")
    print("=" * 65)
    print()
    print("ANSWER (from Gemini):")
    print(answer)

    # Show source chunks
    if show_chunks:
        print()
        print("SOURCE CHUNKS USED (what Gemini actually saw):")
        print("-" * 65)

        for i, chunk in enumerate(deduplicated_chunks): # Use deduplicated_chunks for display
            print(f"Chunk {i+1}:")
            print(chunk.page_content) # Display full content
            print()

    return answer


print("ask_document() function ready.")
print("Pipeline: Question → ChromaDB retrieval → Gemini → Answer")

ask_document() function ready.
Pipeline: Question → ChromaDB retrieval → Gemini → Answer


**Step 9  Test the RAG AI Agent**

In this step, I tested the complete RAG pipeline to confirm that the system works correctly. The uploaded meeting transcript is searched using ChromaDB, the most relevant chunks are retrieved, and Gemini uses only those retrieved sections to generate an accurate summary. This verifies that the AI agent can answer questions based on the document rather than relying on general knowledge.

In [ ]:
ask_document("Summarize this meeting.")

QUESTION: Summarize this meeting.

ANSWER (from Gemini):
Based on the provided transcript, here is a summary of the ABC Technologies Weekly Project Meeting held on July 10, 2026:

**Meeting Objective**
The meeting was held to review the progress of the mobile application development, identify challenges, and prepare for the upcoming product launch.

**Key Discussion Points & Updates**
* **UI/UX Design:** Sarah Johnson reported that the user interface design is approximately 80% complete. She plans to make accessibility improvements, including better color contrast and larger navigation buttons.
* **Testing:** David Brown recommended a final end-to-end system test. Michael Green agreed to organize this final testing session next Friday with the development team.
* **Marketing & Feedback:** Mary Adams noted that marketing expenses are currently within the planned budget. Linda Thomas suggested creating a feedback form for product demonstration attendees, which John Williams approved and 

'Based on the provided transcript, here is a summary of the ABC Technologies Weekly Project Meeting held on July 10, 2026:\n\n**Meeting Objective**\nThe meeting was held to review the progress of the mobile application development, identify challenges, and prepare for the upcoming product launch.\n\n**Key Discussion Points & Updates**\n* **UI/UX Design:** Sarah Johnson reported that the user interface design is approximately 80% complete. She plans to make accessibility improvements, including better color contrast and larger navigation buttons.\n* **Testing:** David Brown recommended a final end-to-end system test. Michael Green agreed to organize this final testing session next Friday with the development team.\n* **Marketing & Feedback:** Mary Adams noted that marketing expenses are currently within the planned budget. Linda Thomas suggested creating a feedback form for product demonstration attendees, which John Williams approved and requested she complete before the demonstration.

In [ ]:
ask_document("Who attended the meeting?")

QUESTION: Who attended the meeting?

ANSWER (from Gemini):
Based on the provided context, the following people attended the meeting:

* **John Williams** – Project Manager  
* **Sarah Johnson** – UI/UX Designer  
* **David Brown** – Software Developer  
* **Mary Adams** – Marketing Lead  
* **Michael Green** – Quality Assurance Engineer  
* **Linda Thomas** – Customer Support Manager

SOURCE CHUNKS USED (what Gemini actually saw):
-----------------------------------------------------------------
Chunk 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing c...

Chunk 2:
ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Location: Conference Room A 
Meeting Type: Weekly Project Progress Meeting 
 
Participants 
 John...

Chunk 3:
John: Please prepare a detailed marketing budget proposal for 

'Based on the provided context, the following people attended the meeting:\n\n* **John Williams** – Project Manager  \n* **Sarah Johnson** – UI/UX Designer  \n* **David Brown** – Software Developer  \n* **Mary Adams** – Marketing Lead  \n* **Michael Green** – Quality Assurance Engineer  \n* **Linda Thomas** – Customer Support Manager'

In [ ]:
ask_document("What tasks were assigned to each team member?")

QUESTION: What tasks were assigned to each team member?

ANSWER (from Gemini):
Based on the provided context, the tasks assigned to each team member are:

*   **David:** Finish payment integration, fix bugs, and complete API security review.
*   **Michael:** Investigate login issue, perform security testing, and organize final system testing.
*   **Mary:** Launch marketing campaign, prepare budget proposal, and organize product demonstration.
*   **Linda:** Complete user documentation, FAQs, tutorial videos, customer support training materials, and create a feedback form before the product demonstration.
*   **John:** Monitor project progress and approve final launch activities.
*   **Sarah:** Make a few accessibility improvements (including better color contrast and larger navigation buttons for visually impaired users).

SOURCE CHUNKS USED (what Gemini actually saw):
-----------------------------------------------------------------
Chunk 1:
 David: Finish payment integration, fix bu

'Based on the provided context, the tasks assigned to each team member are:\n\n*   **David:** Finish payment integration, fix bugs, and complete API security review.\n*   **Michael:** Investigate login issue, perform security testing, and organize final system testing.\n*   **Mary:** Launch marketing campaign, prepare budget proposal, and organize product demonstration.\n*   **Linda:** Complete user documentation, FAQs, tutorial videos, customer support training materials, and create a feedback form before the product demonstration.\n*   **John:** Monitor project progress and approve final launch activities.\n*   **Sarah:** Make a few accessibility improvements (including better color contrast and larger navigation buttons for visually impaired users).'

## Test the updated `ask_document` function with a new question

In [40]:
ask_document("What are the key details regarding the product launch?")

QUESTION: What are the key details regarding the product launch?

ANSWER (from Gemini):
Based on the provided meeting transcript, here are the key details regarding the product launch:

*   **Official Launch Date:** The official application launch date is set for **August 1, 2026**.
*   **Launch Readiness Review:** The team will review final launch readiness at their next meeting on **Monday, July 14, 2026**.
*   **Live Product Demonstration:** A live product demonstration is scheduled for **July 25, 2026**. Linda suggested preparing a feedback form for users attending this demonstration to help identify final improvements before the official launch.
*   **Marketing Campaign Launch:** 
    *   The digital marketing campaign is scheduled to launch "next Tuesday" across Facebook, LinkedIn, Instagram, and X (formerly Twitter).
    *   John emphasized that all promotional materials must clearly communicate the application's key features and launch date.
*   **Preparatory Activities Leading

'Based on the provided meeting transcript, here are the key details regarding the product launch:\n\n*   **Official Launch Date:** The official application launch date is set for **August 1, 2026**.\n*   **Launch Readiness Review:** The team will review final launch readiness at their next meeting on **Monday, July 14, 2026**.\n*   **Live Product Demonstration:** A live product demonstration is scheduled for **July 25, 2026**. Linda suggested preparing a feedback form for users attending this demonstration to help identify final improvements before the official launch.\n*   **Marketing Campaign Launch:** \n    *   The digital marketing campaign is scheduled to launch "next Tuesday" across Facebook, LinkedIn, Instagram, and X (formerly Twitter).\n    *   John emphasized that all promotional materials must clearly communicate the application\'s key features and launch date.\n*   **Preparatory Activities Leading to the Launch:**\n    *   **Final System Testing:** Scheduled for "next Frida

## Creating `app.py` for Streamlit Deployment


**Step 10  Build and Deploy the Streamlit Web Application**

In this step, I converted my RAG system into an interactive Streamlit web application. The app allows users to upload a meeting transcript in PDF format, automatically processes and stores it in ChromaDB, and lets users ask questions about the meeting. Gemini then uses the retrieved document chunks to generate accurate answers, making the AI agent easy to use through a simple web interface.

In [21]:
%%writefile app.py

import streamlit as st
import os

# LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# --- Streamlit UI ---
st.set_page_config(page_title="MeetingMind AI Agent", layout="wide")
st.title("🧠 MeetingMind AI Agent")
st.markdown("--- Say goodbye to information overload. Ask questions, get summaries, and extract key decisions from your meeting transcripts with ease. ---")

# Initialize session state variables if they don't exist
if 'vectorstore' not in st.session_state:
    st.session_state.vectorstore = None
if 'pdf_uploaded' not in st.session_state:
    st.session_state.pdf_uploaded = False

# Gemini API Key input
gemini_api_key = st.secrets.get("GOOGLE_API_KEY", "")

# --- Main Application Logic ---
st.write("### Upload your Meeting Transcript (PDF)")
uploaded_file = st.file_uploader("Choose a PDF file", type="pdf")

if uploaded_file is not None:
    st.session_state.pdf_uploaded = True
    st.success("PDF uploaded successfully!")

    # Save the uploaded file temporarily
    with open("uploaded_meeting.pdf", "wb") as f:
        f.write(uploaded_file.getbuffer())
    doc_filename = "uploaded_meeting.pdf"

    st.write("### Processing Document...")

    # Load the document
    with st.spinner("Loading PDF..."):
        loader = PyPDFLoader(doc_filename)
        documents = loader.load()
        st.success(f"Loaded {len(documents)} pages.")

    # Split into chunks
    with st.spinner("Splitting document into chunks..."):
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=100
        )
        chunks = text_splitter.split_documents(documents)
        st.success(f"Created {len(chunks)} chunks.")

    # Embed chunks and store in ChromaDB
    with st.spinner("Embedding chunks and creating vector store (this may take a moment)..."):
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        st.session_state.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name="my_document"
        )
        st.success("Vector store created successfully!")

    st.write("### Document Ready for Q&A!")

    # Display a text area for questions and a button to ask
    question_input = st.text_area("Ask a question about the meeting transcript:", key="question_area")

    if st.button("Get Answer"):
        if not gemini_api_key:
            st.error("Google API Key not found in Streamlit Secrets.")

        elif st.session_state.vectorstore is None:
            st.error("Please upload a PDF and wait for it to process.")

        else:

            # Setup LLM
            llm = ChatGoogleGenerativeAI(
                model="gemini-3.5-flash", # Updated to a valid model
                google_api_key=gemini_api_key,
                temperature=0
            )

            def ask_document(question):
                retriever = st.session_state.vectorstore.as_retriever(
                    search_kwargs={"k": 5}
                )

                retrieved_chunks = retriever.invoke(question)

                # Deduplicate chunks to avoid sending redundant information
                unique_chunks_content = set()
                deduplicated_chunks = []
                for chunk in retrieved_chunks:
                    if chunk.page_content not in unique_chunks_content:
                        unique_chunks_content.add(chunk.page_content)
                        deduplicated_chunks.append(chunk)

                context = "\n\n---\n\n".join(
                    [chunk.page_content for chunk in deduplicated_chunks]
                )

                user_prompt = f"""You are an AI assistant designed to extract and summarize information from meeting transcripts.
Based on the following CONTEXT from the meeting, please answer the QUESTION.
Your answer should be as comprehensive and informative as possible, drawing all relevant details and making logical inferences *only* from the provided CONTEXT.
If the CONTEXT does not contain enough information to provide a direct answer, or if the information is entirely absent, explain what information is missing or state that the answer cannot be found within the document.
Do not use any external knowledge.

CONTEXT:
{context}

QUESTION:
{question}
"""

                response = llm.invoke(user_prompt)

                # Extract the text content from the response
                answer = response.content[0]["text"]

                return answer, deduplicated_chunks

            with st.spinner("Getting answer from Gemini..."):

                answer, source_chunks = ask_document(question_input)

                st.write("### Answer:")
                st.write(answer)

                st.write("### Source Chunks Used:")

                for i, chunk in enumerate(source_chunks):
                    st.write(f"**Chunk {i+1}:**")
                    st.info(chunk.page_content)


Overwriting app.py


## Creating `requirements.txt`

**Step 11  Create the Project Requirements File**

In this step, I created a requirements.txt file that lists all the Python

In [ ]:
%%writefile requirements.txt

streamlit
langchain
langchain-community
langchain-google-genai
langchain-text-splitters
chromadb
pypdf
sentence-transformers
InstructorEmbedding
torch

Overwriting requirements.txt


**Step 12 – Create the Project Documentation (README)**

In this step, I created a README.md file to document the project. The README provides an overview of the application, its features, the technologies used, installation instructions, and usage information. It helps anyone who visits the GitHub repository understand the purpose of the project and how to run it successfully.

In [ ]:
%%writefile README.md

# MeetingMind AI Agent

MeetingMind AI Agent is a Retrieval-Augmented Generation (RAG) application that allows users to upload meeting transcripts in PDF format and ask questions about the document using Google's Gemini large language model.

## Features

- Upload meeting transcript PDFs
- Automatic text chunking
- Semantic search using ChromaDB
- Embeddings using all-MiniLM-L6-v2
- Question answering with Gemini
- Displays source chunks used to generate answers

## Technologies Used

- Python
- Streamlit
- LangChain
- Google Gemini
- ChromaDB
- HuggingFace Embeddings
- PyPDF

## Installation

```bash
pip install -r requirements.txt

Overwriting README.md


In [ ]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
